# 5 - Transfer Learning (EfficientNetB0)

Use ImageNet-pretrained features to get high accuracy with less overfitting.

**Phase 1**: Train with the backbone frozen, so only the head learns.
**Phase 2**: Unfreeze the top layers and fine-tune at a lower learning rate.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.config import Config
from src.utils import set_global_seed
from src.data_loader import create_all_pipelines, compute_class_weights
from src.models.transfer_learning import build_transfer_model, unfreeze_backbone
from src.training import train_model, compile_model
from src.evaluation import evaluate_model

cfg = Config()
cfg.epochs = 30
cfg.initial_lr = 1e-3
set_global_seed(cfg.seed)

train_ds, val_ds, test_ds = create_all_pipelines(cfg)
class_weights = compute_class_weights(cfg)

## Phase 1 - Frozen Backbone Training

In [ ]:
model = build_transfer_model(cfg, freeze_backbone=True, augment=True)
model.summary()

In [ ]:
history_phase1 = train_model(
    model,
    train_ds,
    val_ds,
    cfg,
    experiment_name='03_efficientnet_frozen',
    class_weight=class_weights,
)

## Phase 2 - Fine-Tuning

In [ ]:
# Unfreeze top 20 layers of the backbone
unfreeze_backbone(model, num_layers_to_unfreeze=20)

# Recompile with a lower learning rate for fine-tuning
cfg.epochs = 20
cfg.initial_lr = 1e-4
compile_model(model, cfg, label_smoothing=0.1)

history_phase2 = train_model(
    model,
    train_ds,
    val_ds,
    cfg,
    experiment_name='04_efficientnet_finetuned',
    class_weight=class_weights,
)

In [ ]:
from src.utils import plot_training_history

plot_training_history(
    cfg.experiment_dir('04_efficientnet_finetuned') / 'history.csv',
    title='Transfer Learning - Fine-Tuning',
)

In [ ]:
loss, acc, top5 = evaluate_model(model, test_ds)